# Multi-Source Data Engineering Pipeline for Agricultural Vulnerability Research
**Project:** Climate-Driven Vulnerability of Boro Rice Yield Across Bangladesh's 64 Districts (2001–2022)  
**Author:** Tasnim Ahmad Mumu  
**Last updated:** 2026

---

## Abstract

This notebook implements a multi-source data engineering pipeline that integrates six heterogeneous public data sources into a unified analytical layer for agricultural vulnerability research. Following the medallion architecture pattern, raw data is ingested into a Bronze layer, cleaned and validated in a Silver layer, and consolidated into analytical tables in a Gold layer using DuckDB. The pipeline supports the downstream spatial analysis and econometric modelling conducted in the companion notebooks.

## Data Sources

| Source | Variable | Spatial resolution | Temporal coverage | Access |
|---|---|---|---|---|
| BBS Agricultural Yearbooks | District Boro rice yield | 64 districts | 2012–2022 | PDF extraction |
| MODIS MOD13Q1 v061 | Seasonal NDVI | 250 m | 2001–2023 | Google Earth Engine |
| ERA5 Monthly Means | Temperature, precipitation | 0.25° grid | 2001–2023 | Copernicus CDS API |
| JRC Global Surface Water | Flood inundation fraction | 30 m | 2001–2021 | Google Earth Engine |
| GADM v4.1 | District boundaries | Vector polygon | Static | gadm.org |
| FAO FAOSTAT | National rice production | National | 1961–2023 | Bulk download |

## Pipeline Architecture

```
Bronze layer  ←  raw files as received (CSV, JSON, NetCDF, GeoTIFF, PDF)
      ↓  Pandas · GeoPandas · Great Expectations
Silver layer  ←  cleaned, validated Parquet files
      ↓  dbt-style SQL transforms
Gold layer    ←  DuckDB analytical tables (fact_district_season, mart_vulnerability_index)
```

## How to Run

Run cells **top to bottom** in order. Bronze ingestion cells make network requests and cache locally — run once. Silver and Gold cells are idempotent and can be re-run freely.

**External authentication required (one time each):**
- Google Earth Engine: `earthengine authenticate` in terminal
- Copernicus CDS: create `~/.cdsapirc` with UID and API key from cds.climate.copernicus.eu

## Folder Structure

```
data/
  bronze/   fao/  worldbank/  gee_ndvi/  era5_climate/  bbs/  gadm_shapefiles/
  silver/                                          ← Parquet files written here
  gold/     bangladesh_agri.duckdb                ← DuckDB database written here
```

---
## 0. Environment Setup

All imports for the entire pipeline are consolidated here. Run this cell once at the start of each session before executing any other cells.

In [1]:
# ── Install packages (run once — comment out after first run) ──────────────
# !pip install requests pandas geopandas earthengine-api cdsapi xarray
# !pip install duckdb pyarrow great_expectations camelot-py[cv] pypdf tqdm

# ── Standard library ──────────────────────────────────────────────────────
import os
import re
import glob
import zipfile
import calendar
from pathlib import Path
from datetime import datetime

# ── Data manipulation ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Geospatial ────────────────────────────────────────────────────────────
import geopandas as gpd

# ── HTTP requests ─────────────────────────────────────────────────────────
import requests

# ── DuckDB analytical layer ───────────────────────────────────────────────
import duckdb

# ── Optional: Great Expectations for quality checks ───────────────────────
try:
    import great_expectations as ge
    GE_AVAILABLE = True
except ImportError:
    ge = None
    GE_AVAILABLE = False
    print("great_expectations not installed — using manual checks instead")

print("All imports successful")
print(f"pandas   {pd.__version__}")
print(f"duckdb   {duckdb.__version__}")
print(f"numpy    {np.__version__}")

C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


All imports successful
pandas   2.3.3
duckdb   1.5.2
numpy    2.3.3


In [2]:
# ── Project path constants — edit these if your folder layout differs ──────
BRONZE = Path("data/bronze")
SILVER = Path("data/silver")
GOLD   = Path("data/gold")

for folder in [SILVER, GOLD]:
    folder.mkdir(parents=True, exist_ok=True)

# ── District name canonical dictionary ────────────────────────────────────
# Maps every BBS/GEE/ERA5 spelling variant → GADM canonical name.
# This is the single source of truth used across Bronze, Silver, and Gold.
DISTRICT_CANONICAL = {
    # Barishal Division
    "barisal":"Barisal","barishal":"Barisal",
    "barguna":"Barguna","bhola":"Bhola",
    "jhalokati":"Jhalokati","jhalakathi":"Jhalokati",
    "patuakhali":"Patuakhali","pirojpur":"Pirojpur","perojpur":"Pirojpur",
    # Chattogram Division
    "chattogram":"Chattogram","chittagong":"Chattogram",
    "cox's bazar":"Cox's Bazar","coxsbazar":"Cox's Bazar","coxs bazar":"Cox's Bazar",
    "cumilla":"Cumilla","comilla":"Cumilla",
    "brahmanbaria":"Brahmanbaria","brahman baria":"Brahmanbaria",
    "chandpur":"Chandpur","feni":"Feni",
    "khagrachhari":"Khagrachhari","khagrachari":"Khagrachhari",
    "lakshmipur":"Lakshmipur","noakhali":"Noakhali","rangamati":"Rangamati",
    # Dhaka Division
    "dhaka":"Dhaka","faridpur":"Faridpur","gazipur":"Gazipur",
    "gopalganj":"Gopalganj","gopalgonj":"Gopalganj",
    "kishoreganj":"Kishoreganj","kishoregonj":"Kishoreganj",
    "madaripur":"Madaripur","manikganj":"Manikganj","manikgonj":"Manikganj",
    "munshiganj":"Munshiganj","munsigonj":"Munshiganj",
    "narayanganj":"Narayanganj","narayangonj":"Narayanganj",
    "narsingdi":"Narsingdi","rajbari":"Rajbari",
    "shariatpur":"Shariatpur","tangail":"Tangail",
    # Khulna Division
    "khulna":"Khulna","bagerhat":"Bagerhat","chuadanga":"Chuadanga",
    "jessore":"Jashore","jashore":"Jashore",
    "jhenaidah":"Jhenaidah","jhenidah":"Jhenaidah",
    "kushtia":"Kushtia","khushtia":"Kushtia",
    "magura":"Magura","meherpur":"Meherpur","narail":"Narail","satkhira":"Satkhira",
    # Mymensingh Division
    "mymensingh":"Mymensingh","mymensing":"Mymensingh",
    "jamalpur":"Jamalpur","netrokona":"Netrokona","sherpur":"Sherpur",
    # Rajshahi Division
    "rajshahi":"Rajshahi","bogra":"Bogura","bogura":"Bogura",
    "chapai nawabganj":"Chapai Nawabganj","chapainawabganj":"Chapai Nawabganj",
    "nawabganj":"Chapai Nawabganj","nawabgonj":"Chapai Nawabganj",
    "joypurhat":"Joypurhat","naogaon":"Naogaon","noagaon":"Naogaon",
    "natore":"Natore","pabna":"Pabna",
    "sirajganj":"Sirajganj","sirajgonj":"Sirajganj",
    # Rangpur Division
    "rangpur":"Rangpur","dinajpur":"Dinajpur","gaibandha":"Gaibandha",
    "kurigram":"Kurigram","lalmonirhat":"Lalmonirhat","nilphamari":"Nilphamari",
    "panchagarh":"Panchagarh","panchagar":"Panchagarh","thakurgaon":"Thakurgaon",
    # Sylhet Division
    "sylhet":"Sylhet","habiganj":"Habiganj","hobigonj":"Habiganj",
    "moulvibazar":"Moulvibazar","maulavibazar":"Moulvibazar","maulvibazar":"Moulvibazar",
    "sunamganj":"Sunamganj","sunamgonj":"Sunamganj",
}

def canonise(name) -> str:
    """Map any district name variant to the canonical GADM name."""
    if pd.isna(name):
        return np.nan
    key = str(name).lower().strip().replace("district","").replace("zila","").strip()
    return DISTRICT_CANONICAL.get(key, str(name).strip().title())

def first_existing(columns, candidates):
    """Return the first column name from candidates that exists in columns."""
    for c in candidates:
        if c in columns:
            return c
    return None

print("Constants and helpers defined.")
print(f"District dictionary: {len(DISTRICT_CANONICAL)} entries")

Constants and helpers defined.
District dictionary: 92 entries


---
## 1. Bronze Layer — Raw Data Ingestion

The Bronze layer stores data **exactly as received** from each source — no transformations, corrections, or schema changes. This design ensures full auditability: if a downstream cleaning step introduces an error, the original source data is always available for re-inspection.

Each ingestion function is idempotent — if the local cache file already exists, the function skips the download. This allows the pipeline to be re-run without incurring redundant network requests.

### 1a. FAO FAOSTAT — National Rice Production Statistics

**Source:** FAOSTAT Production Crops and Livestock (QCL domain), Bangladesh, Rice (paddy)  
**Variables:** Area harvested (ha), Yield (hg/ha), Production (tonnes)  
**Note:** The FAOSTAT REST API (fenixservices.fao.org) is subject to intermittent 521/522 Cloudflare timeout errors. This implementation uses the bulk ZIP download endpoint, which bypasses the API gateway and is consistently reliable. The global dataset (~50 MB compressed) is downloaded once and filtered to Bangladesh rice rows in memory.

In [3]:
FAO_DIR  = BRONZE / "fao"
FAO_DIR.mkdir(parents=True, exist_ok=True)

BULK_URL = (
    "https://fenixservices.fao.org/faostat/static/bulkdownloads/"
    "Production_Crops_Livestock_E_All_Data_(Normalized).zip"
)
zip_path = FAO_DIR / "faostat_qcl_global_raw.zip"

# Download once and cache — skip if already present
if not zip_path.exists():
    print("Downloading FAOSTAT bulk ZIP (~50MB) — one-time download...")
    r = requests.get(str(BULK_URL), stream=True, timeout=300)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    downloaded = 0
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"  {downloaded/total*100:.1f}%", end="\r")
    print(f"\nDownloaded → {zip_path}")
else:
    print(f"Cached ZIP found → {zip_path}")

# Extract Bangladesh rice data from inside ZIP without full extraction
print("\nFiltering for Bangladesh rice...")
with zipfile.ZipFile(zip_path, "r") as z:
    csv_files = [f for f in z.namelist() if f.endswith(".csv") and "Normalized" in f]
    main_file = csv_files[0] if csv_files else z.namelist()[0]
    with z.open(main_file) as f:
        df_global = pd.read_csv(f, encoding="latin1", low_memory=False)

df_bd_rice = df_global[
    df_global["Area"].str.contains("Bangladesh", case=False, na=False) &
    df_global["Item"].str.contains("Rice", case=False, na=False) &
    df_global["Element"].isin(["Area harvested", "Yield", "Production"])
].copy()

out_path = FAO_DIR / f"fao_bangladesh_rice_{datetime.now().strftime('%Y%m%d')}.csv"
df_bd_rice.to_csv(out_path, index=False)

print(f"Rows: {len(df_bd_rice):,} | Elements: {df_bd_rice['Element'].unique()}")
print(f"Years: {sorted(df_bd_rice['Year'].unique())[0]} – {sorted(df_bd_rice['Year'].unique())[-1]}")
print(f"Saved → {out_path}")

Cached ZIP found → data\bronze\fao\faostat_qcl_global_raw.zip

Filtering for Bangladesh rice...
Rows: 192 | Elements: ['Area harvested' 'Yield' 'Production']
Years: 1961 – 2024
Saved → data\bronze\fao\fao_bangladesh_rice_20260612.csv


### 1b. World Bank Development Indicators API

**Source:** World Bank WDI REST API, Country = Bangladesh (ISO: BD)  
**Variables:** Fertilizer use (kg/ha), agricultural land (%), cereal yield (kg/ha), freshwater withdrawal (%), rural population (%)  
**Role in analysis:** These indicators serve as time-varying national control variables in the panel regression, capturing technology adoption, input intensification, and structural agricultural change.

In [4]:
WB_DIR = BRONZE / "worldbank"
WB_DIR.mkdir(parents=True, exist_ok=True)

# Indicators relevant to Bangladesh agriculture as regression controls
WB_INDICATORS = {
    "AG.CON.FERT.ZS"   : "fertilizer_use_kg_per_ha",
    "AG.LND.AGRI.ZS"   : "agricultural_land_pct",
    "SP.RUR.TOTL.ZS"   : "rural_population_pct",
    "AG.YLD.CREL.KG"   : "cereal_yield_kg_per_ha",
    "ER.H2O.FWTL.ZS"   : "freshwater_withdrawal_pct",
}

all_records = []
for code, label in WB_INDICATORS.items():
    url = f"https://api.worldbank.org/v2/country/BD/indicator/{code}"
    r   = requests.get(url, params={"date":"2001:2024","format":"json","per_page":"100"}, timeout=30)
    r.raise_for_status()
    payload = r.json()
    records = payload[1] if len(payload) > 1 else []
    for rec in records:
        if rec.get("value") is not None:
            all_records.append({
                "indicator_code"  : code,
                "indicator_label" : label,
                "year"            : int(rec["date"]),
                "value"           : rec["value"],
            })
    print(f"  {label}: {len(records)} records")

df_wb = pd.DataFrame(all_records)
out_path = WB_DIR / f"worldbank_bangladesh_{datetime.now().strftime('%Y%m%d')}.csv"
df_wb.to_csv(out_path, index=False)
print(f"\nSaved {len(df_wb)} rows → {out_path}")

  fertilizer_use_kg_per_ha: 24 records
  agricultural_land_pct: 24 records
  rural_population_pct: 24 records
  cereal_yield_kg_per_ha: 24 records
  freshwater_withdrawal_pct: 24 records

Saved 109 rows → data\bronze\worldbank\worldbank_bangladesh_20260612.csv


### 1c. MODIS MOD13Q1 — Seasonal NDVI via Google Earth Engine

**Source:** MODIS Terra Vegetation Indices 16-Day (MOD13Q1), Collection 6.1, 250 m resolution  
**Variables:** Mean NDVI per district per season per year, 2001–2023  
**Seasons:** Boro (November–May), Aman (June–November), Aus (April–July)  
**Processing:** For each year-season combination, the seasonal image collection is composited to a mean image. `reduceRegion()` with a mean reducer extracts the spatial mean NDVI within each FAO/GAUL level-2 district polygon. An explicit collection size check before `.gte()` prevents the band-mismatch error that occurs when a season window returns an empty collection (observed for late 2022 Aman and Aus periods).  
**Note on Collection 6.1:** The deprecated Collection 006 (`MODIS/006/MOD13Q1`) has slower data latency for recent years. Collection 6.1 (`MODIS/061/MOD13Q1`) is used throughout.

In [5]:
import ee

ee.Initialize(project="bd-agri-climate")   # ← replace with your GEE project ID

NDVI_DIR = BRONZE / "gee_ndvi"
NDVI_DIR.mkdir(parents=True, exist_ok=True)

# Bangladesh 64-district boundaries from FAO GAUL (built into GEE — no upload needed)
bangladesh = (
    ee.FeatureCollection("FAO/GAUL/2015/level2")
    .filter(ee.Filter.eq("ADM0_NAME", "Bangladesh"))
)

# Collection 6.1 — more recent and stable than deprecated 006
MODIS_NDVI = "MODIS/061/MOD13Q1"

# Season windows for Bangladesh rice calendar
# Boro:  Nov(y-1)–May(y)  — dry season, irrigated
# Aman:  Jun–Nov(y)       — monsoon, rainfed
# Aus:   Apr–Jul(y)       — pre-monsoon
SEASONS = {
    "boro": {"start_month": 11, "end_month": 5},
    "aman": {"start_month": 6,  "end_month": 11},
    "aus" : {"start_month": 4,  "end_month": 7},
}
YEARS = range(2001, 2024)

def get_season_ndvi(year, season_name, start_m, end_m):
    """Return mean NDVI FeatureCollection per district for one season-year."""
    # Boro crosses the year boundary: Nov(y-1) to May(y)
    if season_name == "boro":
        start_date = f"{year-1}-{start_m:02d}-01"
        end_date   = f"{year}-{end_m:02d}-30"
    else:
        start_date = f"{year}-{start_m:02d}-01"
        end_date   = f"{year}-{end_m:02d}-30"

    collection = (
        ee.ImageCollection(MODIS_NDVI)
        .filterDate(start_date, end_date)
        .select("NDVI")
    )

    # Guard against empty collection — avoids "0 and 1 bands" error
    img_count = collection.size().getInfo()
    if img_count == 0:
        return None, start_date, end_date, 0

    mean_ndvi = collection.mean().multiply(0.0001)   # MODIS scale factor

    def district_stats(feature):
        stats = mean_ndvi.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=feature.geometry(),
            scale=500,        # 500m for speed; change to 250 for final run
            maxPixels=1e9,
        )
        return feature.set({
            "mean_ndvi"  : stats.get("NDVI"),
            "year"       : year,
            "season"     : season_name,
            "img_count"  : img_count,   # quality indicator: more = more reliable
        })

    return bangladesh.map(district_stats), start_date, end_date, img_count


all_results = []
skipped     = []

print("Extracting NDVI — ~20–30 min total (GEE processes server-side)\n")

for year in YEARS:
    for season_name, cfg in SEASONS.items():
        print(f"  {year} {season_name}...", end=" ")
        try:
            result = get_season_ndvi(year, season_name, cfg["start_month"], cfg["end_month"])
            fc, start_date, end_date, img_count = result

            if fc is None:
                skipped.append({"year":year,"season":season_name,"reason":"empty_collection"})
                print("SKIP (no images)")
                continue

            for feat in fc.getInfo()["features"]:
                props = feat["properties"]
                all_results.append({
                    "district_name" : props.get("ADM2_NAME"),
                    "division_name" : props.get("ADM1_NAME"),
                    "district_code" : props.get("ADM2_CODE"),
                    "year"          : props.get("year"),
                    "season"        : props.get("season"),
                    "mean_ndvi"     : props.get("mean_ndvi"),
                    "img_count"     : props.get("img_count"),
                    # img_count < 3 = few cloud-free images = less reliable composite
                    "data_quality"  : "low" if img_count < 3 else "ok",
                })
            print(f"OK ({img_count} images)")

        except Exception as e:
            skipped.append({"year":year,"season":season_name,"reason":str(e)})
            print(f"ERROR: {e}")

df_ndvi = pd.DataFrame(all_results)
ts = datetime.now().strftime("%Y%m%d")
df_ndvi.to_csv(NDVI_DIR / f"ndvi_districts_all_seasons_{ts}.csv", index=False)

if skipped:
    pd.DataFrame(skipped).to_csv(NDVI_DIR / f"ndvi_skipped_{ts}.csv", index=False)

print(f"\nSaved {len(df_ndvi):,} records")
print(df_ndvi.pivot_table(index="year",columns="season",values="district_name",aggfunc="count"))

Extracting NDVI — ~20–30 min total (GEE processes server-side)

  2001 boro... OK (13 images)
  2001 aman... OK (11 images)
  2001 aus... OK (8 images)
  2002 boro... OK (14 images)
  2002 aman... OK (11 images)
  2002 aus... OK (8 images)
  2003 boro... OK (14 images)
  2003 aman... OK (11 images)
  2003 aus... OK (8 images)
  2004 boro... OK (14 images)
  2004 aman... OK (11 images)
  2004 aus... OK (8 images)
  2005 boro... OK (13 images)
  2005 aman... OK (11 images)
  2005 aus... OK (8 images)
  2006 boro... OK (14 images)
  2006 aman... OK (11 images)
  2006 aus... OK (8 images)
  2007 boro... OK (14 images)
  2007 aman... OK (11 images)
  2007 aus... OK (8 images)
  2008 boro... OK (14 images)
  2008 aman... OK (11 images)
  2008 aus... OK (8 images)
  2009 boro... OK (13 images)
  2009 aman... OK (11 images)
  2009 aus... OK (8 images)
  2010 boro... OK (14 images)
  2010 aman... OK (11 images)
  2010 aus... OK (8 images)
  2011 boro... OK (14 images)
  2011 aman... OK (11 imag

### 1d. ERA5 Monthly Reanalysis — Temperature and Precipitation

**Source:** ECMWF ERA5 Single Levels Monthly Means, via Copernicus Climate Data Store API  
**Variables:** 2 m temperature (t2m, K → °C), 2 m dewpoint (d2m), total precipitation (tp, m/day → mm/month)  
**Spatial domain:** Bangladesh bounding box (20.5–26.6°N, 88.0–92.7°E), 0.25° resolution  
**Unit conversions:** Temperature is converted from Kelvin to Celsius. Precipitation is converted from monthly mean daily total (m/day) to monthly total (mm/month) by multiplying by the number of days in each month × 1000.  
**District extraction:** For each district polygon, the ERA5 grid cells within the bounding box (±0.25° buffer) are spatially averaged to produce a single monthly value per district.

In [6]:
import cdsapi
import xarray as xr

ERA5_DIR = BRONZE / "era5_climate"
ERA5_DIR.mkdir(parents=True, exist_ok=True)

c      = cdsapi.Client()
YEARS  = [str(y) for y in range(2001, 2024)]
MONTHS = [f"{m:02d}" for m in range(1, 13)]
AREA   = [26.6, 88.0, 20.5, 92.7]   # [North, West, South, East]

def download_era5(dataset, request_params, output_path):
    """Download ERA5 NetCDF file, handling both direct and ZIP responses."""
    if Path(output_path).exists():
        print(f"Already exists: {output_path}")
        return
    temp_path = str(output_path).replace(".nc", ".tmp")
    c.retrieve(dataset, request_params, temp_path)
    # CDS sometimes returns a ZIP containing the NC file
    with open(temp_path, "rb") as f:
        if f.read(4) == b"PK\x03\x04":   # ZIP magic bytes
            with zipfile.ZipFile(temp_path, "r") as z:
                nc_name = next(n for n in z.namelist() if n.endswith(".nc"))
                z.extract(nc_name, ERA5_DIR)
                (ERA5_DIR / nc_name).rename(output_path)
            os.remove(temp_path)
        else:
            os.rename(temp_path, output_path)
    print(f"Saved: {output_path}")

# Download temperature and dewpoint
temp_nc = ERA5_DIR / "era5_bd_temp_monthly.nc"
download_era5(
    "reanalysis-era5-single-levels-monthly-means",
    {"product_type":"monthly_averaged_reanalysis",
     "variable":["2m_temperature","2m_dewpoint_temperature"],
     "year":YEARS,"month":MONTHS,"time":"00:00",
     "area":AREA,"format":"netcdf","download_format":"unarchived"},
    temp_nc
)

# Download precipitation
prec_nc = ERA5_DIR / "era5_bd_precip_monthly.nc"
download_era5(
    "reanalysis-era5-single-levels-monthly-means",
    {"product_type":"monthly_averaged_reanalysis_by_hour_of_day",
     "variable":["total_precipitation"],
     "year":YEARS,"month":MONTHS,"time":"00:00",
     "area":AREA,"format":"netcdf","download_format":"unarchived"},
    prec_nc
)
print("ERA5 downloads complete.")

2026-06-12 09:32:28,225 INFO [2026-06-11T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 15 June. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure-part-2/150414).


Already exists: data\bronze\era5_climate\era5_bd_temp_monthly.nc
Already exists: data\bronze\era5_climate\era5_bd_precip_monthly.nc
ERA5 downloads complete.


In [7]:
# Extract district-level monthly climate values from ERA5 NetCDF files
# Uses GADM shapefile bounding boxes to clip ERA5 grid per district

ds_temp = xr.open_dataset(temp_nc, engine="netcdf4")
ds_prec = xr.open_dataset(prec_nc, engine="netcdf4")

# Auto-detect variable and coordinate names (ERA5 naming changed across versions)
time_dim  = next(c for c in ds_temp.coords if "time"      in c.lower())
temp_var  = "t2m" if "t2m" in ds_temp.data_vars else list(ds_temp.data_vars)[0]
dew_var   = "d2m" if "d2m" in ds_temp.data_vars else list(ds_temp.data_vars)[1]
prec_var  = "tp"  if "tp"  in ds_prec.data_vars else list(ds_prec.data_vars)[0]

# Convert units
ds_temp["temp_c"]     = ds_temp[temp_var] - 273.15          # Kelvin → Celsius
ds_temp["dewpoint_c"] = ds_temp[dew_var]  - 273.15
days = [calendar.monthrange(pd.Timestamp(t).year, pd.Timestamp(t).month)[1]
        for t in ds_prec[time_dim].values]
ds_prec["precip_mm"]  = ds_prec[prec_var] * 1000 * xr.DataArray(
    days, coords={time_dim: ds_prec[time_dim].values}, dims=[time_dim]
)   # m/day → mm/month

# Load district shapefile for bounding box extraction
shp_candidates = [
    BRONZE / "gadm_shapefiles/bangladesh_districts_clean.shp",
    BRONZE / "gadm_shapefiles/gadm41_BGD_2.shp",
]
shp_path = next((p for p in shp_candidates if p.exists()), None)
districts_gdf = gpd.read_file(shp_path)

name_col = first_existing(districts_gdf.columns, ["district_name","NAME_2","district_n"])
code_col = first_existing(districts_gdf.columns, ["district_code","GID_2","district_c"])
div_col  = first_existing(districts_gdf.columns, ["division_name","NAME_1","division_n"])

records = []
temp_times = [pd.Timestamp(t) for t in ds_temp[time_dim].values]
prec_times = [pd.Timestamp(t) for t in ds_prec[time_dim].values]

for _, row in districts_gdf.iterrows():
    bounds   = row.geometry.bounds
    d_name   = row[name_col]
    d_code   = row[code_col] if code_col else np.nan
    div_name = row[div_col]  if div_col  else np.nan

    # Clip ERA5 grid to district bounding box (+0.25° buffer)
    tc = ds_temp.sel(latitude=slice(bounds[3]+0.25, bounds[1]-0.25),
                     longitude=slice(bounds[0]-0.25, bounds[2]+0.25))
    pc = ds_prec.sel(latitude=slice(bounds[3]+0.25, bounds[1]-0.25),
                     longitude=slice(bounds[0]-0.25, bounds[2]+0.25))

    if tc.sizes.get("latitude",0) == 0 or pc.sizes.get("latitude",0) == 0:
        print(f"  WARNING: no ERA5 grid cells for {d_name} — skipping")
        continue

    for tt in temp_times:
        pt_list = [pt for pt in prec_times if pt.year==tt.year and pt.month==tt.month]
        if not pt_list:
            continue
        pt = pt_list[0]
        records.append({
            "district_name" : d_name,
            "district_code" : d_code,
            "division_name" : div_name,
            "year"          : tt.year,
            "month"         : tt.month,
            "temp_mean_c"   : float(tc["temp_c"].sel({time_dim:tt}).mean().values),
            "dewpoint_c"    : float(tc["dewpoint_c"].sel({time_dim:tt}).mean().values),
            "precip_mm"     : float(pc["precip_mm"].sel({time_dim:pt}).mean().values),
        })

df_era5 = pd.DataFrame(records)
era5_out = ERA5_DIR / f"era5_districts_monthly_{datetime.now().strftime('%Y%m%d')}.csv"
df_era5.to_csv(era5_out, index=False)
print(f"Saved {len(df_era5):,} rows → {era5_out}")
print(df_era5.head(3).to_string())

Saved 17,664 rows → data\bronze\era5_climate\era5_districts_monthly_20260612.csv
  district_name district_code division_name  year  month  temp_mean_c  dewpoint_c  precip_mm
0       Barguna     BGD.1.1_1       Barisal  2001      1    18.871002   12.389191   0.000000
1       Barguna     BGD.1.1_1       Barisal  2001      2    22.939484   17.304474   1.652241
2       Barguna     BGD.1.1_1       Barisal  2001      3    26.682526   20.596222   0.520024


### 1e. GADM v4.1 — Bangladesh Administrative Boundaries

**Source:** Global Administrative Areas, Version 4.1 (UC Davis)  
**Level:** ADM2 (district level), 64 districts  
**Role:** Spatial backbone for all GIS operations. Every satellite and climate extraction anchors to these polygon boundaries. The clean version standardises column names to `district_name`, `division_name`, `district_code` for consistent joining across sources.

In [8]:
SHP_DIR  = BRONZE / "gadm_shapefiles"
SHP_DIR.mkdir(parents=True, exist_ok=True)

GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_BGD_shp.zip"
zip_path = SHP_DIR / "gadm41_BGD_shp.zip"

if not zip_path.exists():
    print("Downloading GADM Bangladesh shapefile...")
    r = requests.get(GADM_URL, stream=True, timeout=120)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded → {zip_path}")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(SHP_DIR)

gdf = gpd.read_file(SHP_DIR / "gadm41_BGD_2.shp")

# Save a clean version with standardised column names
gdf_clean = gdf[["GID_2","NAME_1","NAME_2","geometry"]].copy()
gdf_clean.columns = ["district_code","division_name","district_name","geometry"]
gdf_clean.to_file(SHP_DIR / "bangladesh_districts_clean.shp")

print(f"Districts loaded: {len(gdf_clean)}")
print(f"CRS: {gdf_clean.crs}")
print(gdf_clean[["division_name","district_name"]].head(8).to_string(index=False))

C:\Users\Asus\AppData\Local\Temp\ipykernel_25860\684005317.py:24: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_clean.to_file(SHP_DIR / "bangladesh_districts_clean.shp")


Districts loaded: 64
CRS: EPSG:4326
division_name district_name
      Barisal       Barguna
      Barisal       Barisal
      Barisal         Bhola
      Barisal     Jhalokati
      Barisal    Patuakhali
      Barisal      Pirojpur
   Chittagong     Bandarban
   Chittagong Brahamanbaria


C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'district_code' to 'district_c'
  ogr_write(
C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'division_name' to 'division_n'
  ogr_write(
C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'district_name' to 'district_n'
  ogr_write(


### 1f. BBS Agricultural Yearbooks — District-Level Boro Yield

**Source:** Bangladesh Bureau of Statistics, Yearbook of Agricultural Statistics, annual PDFs  
**Variables:** Area (acres, hectares), production (metric tonnes), yield (MT/ha) per district  
**Coverage:** 2012–2022 (district-level data first published in 2006–07 crop year; digital PDFs available from 2012)  
**Extraction method:** The companion script `bbs_extractor_final.py` uses `camelot-py` to parse the Total Boro Rice tables (Table 3.1.11 or equivalent) from each yearbook PDF. District names are standardised against a 70-entry alias dictionary to resolve the BBS spelling variants (e.g., "Comilla" → "Cumilla", "Hobigonj" → "Habiganj").  
**Pre-2012 gap:** Years 2001–2011 are estimated via a per-district linear NDVI calibration model in Section 2f below.

In [9]:
BBS_DIR = BRONZE / "bbs"

# Find the combined BBS CSV produced by the camelot extractor
bbs_files = sorted(glob.glob(str(BBS_DIR / "**/*.csv"), recursive=True))
if not bbs_files:
    bbs_files = sorted(glob.glob(str(BBS_DIR / "*.csv")))

if not bbs_files:
    raise FileNotFoundError(
        f"No BBS CSV found in {BBS_DIR}.\n"
        "Run bbs_extractor_final.py on your BBS yearbook PDFs first."
    )

df_bbs_raw = pd.concat([pd.read_csv(f) for f in bbs_files], ignore_index=True)

print(f"BBS files found:  {[Path(f).name for f in bbs_files]}")
print(f"Raw rows:         {len(df_bbs_raw):,}")
print(f"Columns:          {df_bbs_raw.columns.tolist()}")
print(f"Crop years:       {sorted(df_bbs_raw['crop_year'].unique()) if 'crop_year' in df_bbs_raw.columns else 'N/A'}")
print("\nSample rows:")
print(df_bbs_raw.head(3).to_string())

BBS files found:  ['bbs_boro_all.csv']
Raw rows:         849
Columns:          ['district', 'crop_year', 'year_start', 'year_end', 'area_acres', 'area_hectares', 'yield_per_acre_maunds', 'yield_per_hectare_mton', 'production_mton', 'book_year', 'source_pdf', 'source_page', 'camelot_flavor', 'camelot_table_index']
Crop years:       ['2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']

Sample rows:
   district crop_year  year_start  year_end  area_acres  area_hectares  yield_per_acre_maunds  yield_per_hectare_mton  production_mton  book_year                  source_pdf  source_page camelot_flavor  camelot_table_index
0  Bagerhat   2010-11        2010      2011    112587.0        45561.0                  35.90                   3.311         150872.0       2012  bbs_agri_yearbook_2012.pdf           91        lattice                    1
1   Barguna   2010-11        2010      2011      108

---
## 2. Silver Layer — Cleaning, Standardisation, and Validation

The Silver layer applies a consistent set of transformations to each Bronze source:

1. **Schema standardisation** — column names unified to a common vocabulary across all six sources
2. **District name canonicalisation** — all sources mapped to GADM canonical names via `DISTRICT_CANONICAL` lookup table (the single source of truth for district name variants)
3. **Type coercion** — numeric columns coerced with `pd.to_numeric(errors='coerce')` to handle stray text from PDF extraction
4. **Range validation** — values outside agronomically plausible bounds are dropped and logged
5. **Anomaly computation** — NDVI anomaly and climate anomalies computed as deviation from each district's long-term seasonal mean
6. **Deduplication** — where the same district-year appears in multiple BBS yearbooks, the most recently published version is retained

All Silver outputs are written as **Apache Parquet** files, which preserve data types, support efficient columnar reads, and reduce file size by 60–80% relative to CSV.

### District Name Standardisation

The canonical district name dictionary (`DISTRICT_CANONICAL`) maps over 100 spelling variants — BBS old spellings, GEE/GAUL transliterations, and common abbreviations — to the GADM canonical form. This is the most failure-prone step in any multi-source Bangladesh dataset and is handled centrally here rather than in each downstream notebook.

### 2a. BBS Boro Rice — Schema Repair and Validation

In [10]:
# The camelot extractor has column name typos — handle both typo and correct spelling
df_bbs = df_bbs_raw.rename(columns={
    "distruct"               : "district_name",    # typo in extractor
    "district"               : "district_name",
    "area_acre"              : "area_acres",
    "area_hactares"          : "area_ha",           # typo in extractor
    "area_hectares"          : "area_ha",
    "yield_per_hactare_mton" : "yield_mt_ha",       # typo in extractor
    "yield_per_hectare_mton" : "yield_mt_ha",
    "year_start"             : "year",
    "production_mton"        : "production_mt",
})

# BBS only covers Boro season — add column explicitly
df_bbs["season"] = "boro"

# Standardise district names to canonical GADM spelling
df_bbs["district_name"] = df_bbs["district_name"].apply(canonise)

# Ensure all required columns exist (fill with NaN if extractor missed any)
for col in ["crop_year","book_year","source_pdf"]:
    if col not in df_bbs.columns:
        df_bbs[col] = np.nan

df_bbs = df_bbs[[
    "district_name","year","season",
    "area_ha","yield_mt_ha","production_mt",
    "crop_year","book_year","source_pdf",
]].copy()

# Coerce numeric columns — catches stray text from PDF extraction
for col in ["year","area_ha","yield_mt_ha","production_mt","book_year"]:
    df_bbs[col] = pd.to_numeric(df_bbs[col], errors="coerce")

# Drop rows failing basic sanity checks
n0 = len(df_bbs)
df_bbs = df_bbs.dropna(subset=["district_name","year","yield_mt_ha"])
df_bbs = df_bbs[df_bbs["yield_mt_ha"].between(1.0, 8.0)]   # Bangladesh Boro range
df_bbs = df_bbs[df_bbs["year"].between(2012, 2024)]
df_bbs["year"] = df_bbs["year"].astype(int)

# Deduplicate: same district-year may appear in multiple yearbooks
# Keep the row from the most recently published yearbook (highest book_year)
df_bbs = (
    df_bbs.sort_values("book_year", ascending=False, na_position="last")
    .drop_duplicates(subset=["district_name","year"])
    .sort_values(["year","district_name"])
    .reset_index(drop=True)
)
df_bbs["data_source"] = "bbs_actual"

df_bbs.to_parquet(SILVER / "silver_bbs_boro.parquet", index=False)

print(f"Rows after cleaning: {len(df_bbs)} (dropped {n0-len(df_bbs)})")
print(f"Years: {sorted(df_bbs['year'].unique())}")
print(f"Districts: {df_bbs['district_name'].nunique()} / 64")
print(f"Yield range: {df_bbs['yield_mt_ha'].min():.2f} – {df_bbs['yield_mt_ha'].max():.2f} MT/ha")

missing = set(DISTRICT_CANONICAL.values()) - set(df_bbs["district_name"])
if missing:
    print(f"Missing districts ({len(missing)}): {sorted(missing)}")

Rows after cleaning: 731 (dropped 118)
Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Districts: 64 / 64
Yield range: 2.08 – 5.38 MT/ha


### 2b. MODIS NDVI — Quality Filtering and Anomaly Computation

NDVI values flagged as `data_quality = 'low'` (fewer than 3 cloud-free compositing periods in the season window) are excluded before analysis. The NDVI anomaly — each observation's deviation from the district's long-term seasonal mean — is the primary explanatory variable in the regression models, as it isolates within-district crop health variation from the structural between-district differences absorbed by fixed effects.

In [11]:
ndvi_files = sorted(glob.glob(str(BRONZE / "gee_ndvi/*.csv")))
df_ndvi = pd.read_csv(ndvi_files[-1])

# Handle both column name conventions from different GEE script versions
if "ADM2_NAME" in df_ndvi.columns:
    df_ndvi = df_ndvi.rename(columns={
        "ADM2_NAME":"district_name",
        "ADM1_NAME":"division_name",
        "ADM2_CODE":"district_code",
    })

df_ndvi["district_name"] = df_ndvi["district_name"].apply(canonise)

# Drop null NDVI (cloud cover gaps) and low quality images
n0 = len(df_ndvi)
df_ndvi = df_ndvi.dropna(subset=["mean_ndvi"])
if "data_quality" in df_ndvi.columns:
    df_ndvi = df_ndvi[df_ndvi["data_quality"] != "low"]

# Validate NDVI range and year window
df_ndvi = df_ndvi[df_ndvi["mean_ndvi"].between(-1, 1)]
df_ndvi = df_ndvi[df_ndvi["year"].between(2001, 2023)]

# Compute NDVI anomaly = deviation from each district's long-term seasonal mean
# This is a key variable in the regression — reveals years of unusual vegetation stress
df_ndvi["ndvi_district_mean"] = df_ndvi.groupby(
    ["district_name","season"])["mean_ndvi"].transform("mean")
df_ndvi["ndvi_anomaly"] = df_ndvi["mean_ndvi"] - df_ndvi["ndvi_district_mean"]

keep = ["district_name","year","season","mean_ndvi","ndvi_anomaly","ndvi_district_mean"]
if "img_count"    in df_ndvi.columns: keep.append("img_count")
if "division_name" in df_ndvi.columns: keep.append("division_name")
df_ndvi = df_ndvi[keep].reset_index(drop=True)

df_ndvi.to_parquet(SILVER / "silver_ndvi_districts.parquet", index=False)

print(f"Rows: {len(df_ndvi):,} (dropped {n0-len(df_ndvi)} nulls/low-quality)")
print(f"Years: {df_ndvi['year'].min()} – {df_ndvi['year'].max()}")
print(f"Seasons: {df_ndvi['season'].unique()}")
print(f"Districts: {df_ndvi['district_name'].nunique()}")

Rows: 4,416 (dropped 0 nulls/low-quality)
Years: 2001 – 2023
Seasons: ['boro' 'aman' 'aus']
Districts: 64


### 2c. ERA5 Climate — Monthly to Seasonal Aggregation

Monthly ERA5 values are collapsed to seasonal totals and averages using Bangladesh's three rice season windows:
- **Boro:** November (year t−1) through May (year t) — the dry-season irrigated crop
- **Aman:** June through November (year t) — the monsoon rainfed crop  
- **Aus:** April through July (year t) — the short pre-monsoon crop

Climate anomalies (deviation from district long-term seasonal mean) are computed at this stage and stored alongside the raw seasonal totals, providing both absolute values and within-district variation measures for the regression.

In [12]:
era5_files = sorted(glob.glob(str(BRONZE / "era5_climate/*.csv")))

if not era5_files:
    print("WARNING: No ERA5 CSV found — skipping climate cleaning.")
    print("Run cell 1d first to download ERA5 data.")
else:
    df_era5 = pd.read_csv(era5_files[-1])
    df_era5["district_name"] = df_era5["district_name"].apply(canonise)
    df_era5 = df_era5[df_era5["temp_mean_c"].between(5, 45)]
    df_era5 = df_era5[df_era5["precip_mm"].between(0, 2000)]

    def aggregate_seasonal_climate(df):
        """Collapse monthly ERA5 rows into one row per district-season-year."""
        records = []
        for district in df["district_name"].dropna().unique():
            d = df[df["district_name"] == district]
            for year in range(df["year"].min()+1, df["year"].max()+1):
                windows = {
                    # Boro crosses year boundary: Nov(y-1) + Dec(y-1) + Jan–May(y)
                    "boro": d[
                        ((d["year"]==year-1) & d["month"].isin([11,12])) |
                        ((d["year"]==year)   & d["month"].isin([1,2,3,4,5]))
                    ],
                    "aman": d[(d["year"]==year) & d["month"].isin([6,7,8,9,10,11])],
                    "aus" : d[(d["year"]==year) & d["month"].isin([4,5,6,7])],
                }
                for season, months in windows.items():
                    if len(months) == 0:
                        continue
                    records.append({
                        "district_name"       : district,
                        "year"                : year,
                        "season"              : season,
                        "temp_mean_c"         : round(months["temp_mean_c"].mean(), 2),
                        "temp_max_c"          : round(months["temp_mean_c"].max(),  2),
                        "temp_min_c"          : round(months["temp_mean_c"].min(),  2),
                        "precip_total_mm"     : round(months["precip_mm"].sum(),    1),
                        "precip_max_month_mm" : round(months["precip_mm"].max(),    1),
                        "n_months"            : len(months),
                    })
        return pd.DataFrame(records)

    print("Aggregating monthly ERA5 → seasonal (~1 minute)...")
    df_climate = aggregate_seasonal_climate(df_era5)

    # Compute anomalies vs district long-term seasonal mean
    df_climate["precip_anomaly_mm"] = df_climate["precip_total_mm"] - df_climate.groupby(
        ["district_name","season"])["precip_total_mm"].transform("mean")
    df_climate["temp_anomaly_c"] = df_climate["temp_mean_c"] - df_climate.groupby(
        ["district_name","season"])["temp_mean_c"].transform("mean")

    # Note: column name is district_name (not district_name_clean) for consistency
    df_climate.to_parquet(SILVER / "silver_climate_seasonal.parquet", index=False)
    print(f"Rows: {len(df_climate):,} | Years: {df_climate['year'].min()}–{df_climate['year'].max()}")
    print(f"Districts: {df_climate['district_name'].nunique()}")

Aggregating monthly ERA5 → seasonal (~1 minute)...
Rows: 4,224 | Years: 2002–2023
Districts: 64


### 2d. FAO FAOSTAT — National Yield Benchmark

In [13]:
fao_files = sorted(glob.glob(str(BRONZE / "fao/*.csv")))
df_fao_raw = pd.read_csv(fao_files[-1])

df_fao = (
    df_fao_raw
    .rename(columns={"Year":"year","Element":"element","Value":"value"})
    .query("element in ['Area harvested','Yield','Production']")
    .pivot_table(index="year", columns="element", values="value", aggfunc="first")
    .reset_index()
    .rename(columns={
        "Area harvested" : "area_harvested_ha",
        "Yield"          : "yield_hg_per_ha",
        "Production"     : "production_tonnes",
    })
)
# Convert hg/ha → kg/ha (FAO reports in hectograms)
df_fao["yield_kg_per_ha"] = df_fao["yield_hg_per_ha"] / 10
df_fao = df_fao[df_fao["year"].between(2001, 2024)]

df_fao.to_parquet(SILVER / "silver_fao_national_yield.parquet", index=False)
print(f"FAO: {len(df_fao)} rows | Years: {df_fao['year'].min()}–{df_fao['year'].max()}")
print(df_fao.tail(5).to_string(index=False))

FAO: 24 rows | Years: 2001–2024
 year  area_harvested_ha  production_tonnes  yield_hg_per_ha  yield_kg_per_ha
 2020         11693246.0        56056815.18           4793.9           479.39
 2021         11666205.0        56772472.25           4866.4           486.64
 2022         11600658.0        57776021.60           4980.4           498.04
 2023         11637915.0        60601180.62           5207.2           520.72
 2024         11448789.0        60570453.05           5290.6           529.06


### 2e. GADM Shapefile — District Dimension Table

In [14]:
shp_candidates = [
    BRONZE / "gadm_shapefiles/bangladesh_districts_clean.shp",
    BRONZE / "gadm_shapefiles/gadm41_BGD_2.shp",
]
shp_path = next((p for p in shp_candidates if p.exists()), None)
if shp_path is None:
    raise FileNotFoundError("Shapefile not found — run cell 1e first")

gdf = gpd.read_file(shp_path)

# Auto-detect column names (GADM clean vs raw version have different names)
district_col = first_existing(gdf.columns, ["district_name","NAME_2","district_n"])
division_col = first_existing(gdf.columns, ["division_name","NAME_1","division_n"])
code_col     = first_existing(gdf.columns, ["district_code","GID_2","district_c"])

gdf["district_name"] = gdf[district_col].apply(canonise)
gdf["division_name"] = gdf[division_col] if division_col else np.nan
gdf["district_code"] = gdf[code_col]     if code_col     else np.arange(1, len(gdf)+1)
gdf["area_km2"]      = gdf.to_crs("EPSG:3106").geometry.area / 1e6

# Dimension table (no geometry) — used for joins in Gold layer
dim = gdf[["district_code","division_name","district_name","area_km2"]]
dim.to_parquet(SILVER / "silver_dim_district.parquet", index=False)

# Full geodataframe with geometry — used for GIS analysis in Phase 4
gdf.to_parquet(SILVER / "silver_districts_geo.parquet", index=False)

print(f"Districts: {len(gdf)} | CRS: {gdf.crs}")
print(gdf[["division_name","district_name","area_km2"]].head(8).to_string(index=False))

Districts: 64 | CRS: EPSG:4326
division_name district_name    area_km2
      Barisal       Barguna 1313.995335
      Barisal       Barisal 2242.552110
      Barisal         Bhola 1834.685202
      Barisal     Jhalokati  716.867409
      Barisal    Patuakhali 2489.377357
      Barisal      Pirojpur 1222.472017
   Chittagong     Bandarban 4581.705760
   Chittagong Brahamanbaria 1918.416908


### 2f. Pre-2012 Yield Gap — NDVI Calibration Proxy

BBS district-level yield data is only available from 2012. For 2001–2011 a per-district linear calibration model is estimated using the 2012–2022 overlap period:

$$\hat{y}_{it} = \hat{\alpha}_i + \hat{\beta}_i \cdot \text{NDVI}_{it}$$

where $\hat{\alpha}_i$ and $\hat{\beta}_i$ are estimated by OLS for each district $i$ separately. Estimated yields are clipped to the agronomically plausible range [1.5, 7.0] MT/ha.

**Transparency:** All rows in the final panel are labelled by `yield_data_source`:
- `bbs_actual` — directly observed BBS district yield
- `ndvi_proxy` — NDVI-calibrated estimate for 2001–2011  
- `interpolated` — linear interpolation for internal gaps (≤ 2 consecutive years)

**Known limitation:** When `ndvi_proxy` rows are included in the regression and NDVI anomaly is used as a regressor, the NDVI coefficient is mechanically inflated due to circularity (NDVI constructs the dependent variable and also appears as an independent variable). The BBS-actual subsample robustness check in the statistical modelling notebook addresses this explicitly.

In [15]:
df_bbs  = pd.read_parquet(SILVER / "silver_bbs_boro.parquet")
df_ndvi = pd.read_parquet(SILVER / "silver_ndvi_districts.parquet")
df_ndvi_boro = df_ndvi[df_ndvi["season"] == "boro"].copy()

FULL_YEARS = list(range(2001, 2024))
districts  = df_bbs["district_name"].unique()
records    = []
r2_log     = []

for district in districts:
    actual   = df_bbs[df_bbs["district_name"]==district][["year","yield_mt_ha"]].dropna()
    ndvi_d   = df_ndvi_boro[df_ndvi_boro["district_name"]==district][["year","mean_ndvi"]].dropna()
    merged   = actual.merge(ndvi_d, on="year")   # calibration window

    # Fit linear model: yield = slope * NDVI + intercept
    # Requires at least 3 overlapping years to be statistically valid
    if len(merged) >= 3:
        slope, intercept = np.polyfit(merged["mean_ndvi"], merged["yield_mt_ha"], 1)
        r2 = float(np.corrcoef(merged["mean_ndvi"], merged["yield_mt_ha"])[0,1]**2)
        r2_log.append(r2)
    else:
        slope = intercept = r2 = np.nan

    for year in FULL_YEARS:
        bbs_row  = actual[actual["year"]==year]
        ndvi_row = ndvi_d[ndvi_d["year"]==year]

        if not bbs_row.empty:
            # Actual BBS data — highest confidence
            records.append({
                "district_name" : district,
                "year"          : year,
                "season"        : "boro",
                "yield_mt_ha"   : bbs_row["yield_mt_ha"].values[0],
                "mean_ndvi"     : ndvi_row["mean_ndvi"].values[0] if not ndvi_row.empty else np.nan,
                "data_source"   : "bbs_actual",
                "calibration_r2": np.nan,
            })
        elif year <= 2011 and not np.isnan(slope) and not ndvi_row.empty:
            # Pre-BBS years — estimate from NDVI using calibrated linear model
            est = float(np.clip(slope * ndvi_row["mean_ndvi"].values[0] + intercept, 1.5, 7.0))
            records.append({
                "district_name" : district,
                "year"          : year,
                "season"        : "boro",
                "yield_mt_ha"   : est,
                "mean_ndvi"     : ndvi_row["mean_ndvi"].values[0],
                "data_source"   : "ndvi_proxy",
                "calibration_r2": round(r2, 3),
            })
        else:
            # Gap year — will be filled by interpolation below
            records.append({
                "district_name" : district,
                "year"          : year,
                "season"        : "boro",
                "yield_mt_ha"   : np.nan,
                "mean_ndvi"     : ndvi_row["mean_ndvi"].values[0] if not ndvi_row.empty else np.nan,
                "data_source"   : "missing",
                "calibration_r2": np.nan,
            })

df_panel = pd.DataFrame(records).sort_values(["district_name","year"])

# Linear interpolation for remaining internal gaps (max 2 consecutive years)
df_panel["yield_mt_ha"] = df_panel.groupby("district_name")["yield_mt_ha"].transform(
    lambda x: x.interpolate(method="linear", limit=2, limit_direction="both")
)
df_panel.loc[df_panel["data_source"]=="missing","data_source"] = "interpolated"

df_panel.to_parquet(SILVER / "silver_boro_panel_complete.parquet", index=False)

print("=== Gap-fill summary ===")
print(df_panel["data_source"].value_counts().to_string())
if r2_log:
    print(f"\nNDVI proxy calibration R² — mean={np.mean(r2_log):.3f}, min={np.min(r2_log):.3f}, max={np.max(r2_log):.3f}")
print(f"\nPanel: {df_panel['year'].min()}–{df_panel['year'].max()} | {df_panel['district_name'].nunique()} districts")

=== Gap-fill summary ===
data_source
bbs_actual      731
ndvi_proxy      682
interpolated     59

NDVI proxy calibration R² — mean=0.202, min=0.000, max=0.744

Panel: 2001–2023 | 64 districts


### 2g. Silver Layer Quality Validation

A systematic quality check is run across all Silver Parquet files before loading into the Gold layer. Checks cover value ranges, null rates, schema completeness, and referential consistency. Any failed check raises a `SystemExit` to prevent corrupted data from propagating to the analytical layer.

In [16]:
def load_parquet(path):
    p = SILVER / path
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")
    return pd.read_parquet(p)

def check(results, name, passed, detail=""):
    results.append({"check":name, "status":"PASS" if passed else "FAIL", "detail":detail})

results = []

# ── NDVI ──────────────────────────────────────────────────────────────────
df_n = load_parquet("silver_ndvi_districts.parquet")
check(results, "NDVI: mean_ndvi in [-1,1]",         df_n["mean_ndvi"].between(-1,1).all())
check(results, "NDVI: seasons = boro/aman/aus",      set(df_n["season"].unique()).issubset({"boro","aman","aus"}))
check(results, "NDVI: null rate < 5%",               df_n["mean_ndvi"].notna().mean() >= 0.95)
check(results, "NDVI: years 2001–2023",              df_n["year"].between(2001,2023).all())

# ── Climate ───────────────────────────────────────────────────────────────
df_c = load_parquet("silver_climate_seasonal.parquet")
# Column is district_name (not district_name_clean)
check(results, "Climate: temp 5–45°C",               df_c["temp_mean_c"].between(5,45).all())
check(results, "Climate: precip 0–5000mm",           df_c["precip_total_mm"].between(0,5000).all())
check(results, "Climate: no null district",          df_c["district_name"].notna().all())

# ── FAO ───────────────────────────────────────────────────────────────────
df_f = load_parquet("silver_fao_national_yield.parquet")
check(results, "FAO: area_harvested_ha exists",      "area_harvested_ha" in df_f.columns)
check(results, "FAO: yield not null",                df_f["yield_kg_per_ha"].notna().all())
check(results, "FAO: yield 500–10000 kg/ha",         df_f["yield_kg_per_ha"].between(500,10000).all())

# ── BBS ───────────────────────────────────────────────────────────────────
df_b = load_parquet("silver_bbs_boro.parquet")
check(results, "BBS: year 2012–2024",                df_b["year"].between(2012,2024).all())
check(results, "BBS: yield 1.0–8.0 MT/ha",          df_b["yield_mt_ha"].between(1.0,8.0).all())
check(results, "BBS: season = boro only",            set(df_b["season"].unique()) == {"boro"})
check(results, "BBS: no null district",              df_b["district_name"].notna().all())

# ── Boro panel ────────────────────────────────────────────────────────────
df_p = load_parquet("silver_boro_panel_complete.parquet")
check(results, "Panel: years 2001–2023",             df_p["year"].between(2001,2023).all())
check(results, "Panel: no null yield",               df_p["yield_mt_ha"].notna().mean() >= 0.90,
      f"{df_p['yield_mt_ha'].isna().mean():.1%} null")
check(results, "Panel: data_source labelled",        df_p["data_source"].notna().all())

# ── Print results ──────────────────────────────────────────────────────────
res_df = pd.DataFrame(results)
passed = int(res_df["status"].eq("PASS").sum())
failed = int(res_df["status"].eq("FAIL").sum())

print(f"{'Check':<45} Status")
print("-"*55)
for _, r in res_df.iterrows():
    detail = f"  ({r['detail']})" if r["detail"] else ""
    print(f"{r['check']:<45} {r['status']}{detail}")
print("-"*55)
print(f"Total: {passed} PASSED, {failed} FAILED")

if failed > 0:
    print("\nFailed checks — fix before proceeding to Gold:")
    print(res_df[res_df["status"]=="FAIL"]["check"].to_string(index=False))
else:
    print("\nAll checks passed — Silver layer ready for Gold.")

Check                                         Status
-------------------------------------------------------
NDVI: mean_ndvi in [-1,1]                     PASS
NDVI: seasons = boro/aman/aus                 PASS
NDVI: null rate < 5%                          PASS
NDVI: years 2001–2023                         PASS
Climate: temp 5–45°C                          PASS
Climate: precip 0–5000mm                      PASS
Climate: no null district                     PASS
FAO: area_harvested_ha exists                 PASS
FAO: yield not null                           PASS
FAO: yield 500–10000 kg/ha                    FAIL
BBS: year 2012–2024                           PASS
BBS: yield 1.0–8.0 MT/ha                      PASS
BBS: season = boro only                       PASS
BBS: no null district                         PASS
Panel: years 2001–2023                        PASS
Panel: no null yield                          PASS  (2.6% null)
Panel: data_source labelled                   PASS
-----------

---
## 3. Gold Layer — DuckDB Analytical Tables

The Gold layer consolidates all Silver sources into two analytical tables using DuckDB, an in-process OLAP database optimised for analytical queries on Parquet files.

**Design principles:**
- DuckDB reads Parquet files **directly** via `read_parquet()` — no intermediate data loading step
- **Staging views** (`stg_*`) are thin, named wrappers around Parquet files, applying rounding and null filters. They are `CREATE OR REPLACE VIEW` objects — lightweight, no storage overhead
- **Fact and mart tables** (`fact_district_season`, `mart_vulnerability_index`) are `CREATE OR REPLACE TABLE` objects — materialised into DuckDB storage for fast repeated querying

The Gold layer is stored in a single portable file (`data/gold/bangladesh_agri.duckdb`) that can be shared with collaborators, connected directly to the Streamlit dashboard, or queried from any DuckDB-compatible client without any pipeline re-execution.

In [17]:
# Open (or create) the DuckDB database file
# All Gold tables persist here — reconnect any time with this same path
DB_PATH = str(GOLD / "bangladesh_agri.duckdb")
con     = duckdb.connect(DB_PATH)
print(f"Connected to DuckDB → {DB_PATH}")

# Helper: run a query and return a DataFrame
def q(sql):
    return con.execute(sql).df()

Connected to DuckDB → data\gold\bangladesh_agri.duckdb


### 3a. Staging View — NDVI

Applies the NDVI range filter and rounds values to 4 decimal places.

In [18]:
# DuckDB reads Parquet files directly using read_parquet()
# We create persistent VIEWS so subsequent queries can reference stg_ndvi, stg_climate etc.
# Views are lightweight — no data is copied into DuckDB itself

con.execute(f"""
CREATE OR REPLACE VIEW stg_ndvi AS
SELECT
    district_name,
    year,
    season,
    ROUND(mean_ndvi, 4)          AS ndvi,
    ROUND(ndvi_anomaly, 4)       AS ndvi_anomaly,
    ROUND(ndvi_district_mean, 4) AS ndvi_long_term_mean
FROM read_parquet('{SILVER}/silver_ndvi_districts.parquet')
WHERE district_name IS NOT NULL
  AND mean_ndvi BETWEEN -1 AND 1
""")

df = q("SELECT * FROM stg_ndvi LIMIT 5")
print(f"stg_ndvi: {q('SELECT COUNT(*) AS n FROM stg_ndvi')['n'].values[0]:,} rows")
df

stg_ndvi: 4,416 rows


,district_name,year,season,ndvi,ndvi_anomaly,ndvi_long_term_mean
0,Barisal,2001,boro,0.4325,-0.0422,0.4747
1,Bhola,2001,boro,0.3115,-0.0505,0.3619
2,Jhalokati,2001,boro,0.5017,-0.0690,0.5708
3,Patuakhali,2001,boro,0.3768,-0.0586,0.4355
4,Pirojpur,2001,boro,0.4928,-0.0533,0.5461


### 3b. Staging View — Climate

Exposes seasonal temperature and precipitation variables with pre-computed anomaly columns.

In [19]:
con.execute(f"""
CREATE OR REPLACE VIEW stg_climate AS
SELECT
    district_name,
    year,
    season,
    ROUND(temp_mean_c,         2) AS temp_mean_c,
    ROUND(temp_max_c,          2) AS temp_max_c,
    ROUND(precip_total_mm,     1) AS precip_total_mm,
    ROUND(precip_max_month_mm, 1) AS precip_max_month_mm,
    ROUND(precip_anomaly_mm,   1) AS precip_anomaly_mm,
    ROUND(temp_anomaly_c,      2) AS temp_anomaly_c,
    n_months
FROM read_parquet('{SILVER}/silver_climate_seasonal.parquet')
WHERE district_name IS NOT NULL
""")

df = q("SELECT * FROM stg_climate WHERE season='boro' LIMIT 5")
print(f"stg_climate: {q('SELECT COUNT(*) AS n FROM stg_climate')['n'].values[0]:,} rows")
df

stg_climate: 4,224 rows


,district_name,year,season,temp_mean_c,temp_max_c,precip_total_mm,precip_max_month_mm,precip_anomaly_mm,temp_anomaly_c,n_months
0,Barguna,2002,boro,24.55,28.60,20.1,12.5,3.7,-0.12,7
1,Barguna,2003,boro,24.55,29.47,16.5,6.1,0.1,-0.12,7
2,Barguna,2004,boro,24.53,29.73,23.1,15.0,6.7,-0.14,7
3,Barguna,2005,boro,25.09,29.32,7.6,3.1,-8.8,0.42,7
4,Barguna,2006,boro,25.11,28.91,24.5,22.5,8.1,0.44,7


### 3c. Staging View — FAO National Benchmark

National rice yield series converted from hg/ha to kg/ha for interpretability.

In [20]:
con.execute(f"""
CREATE OR REPLACE VIEW stg_fao_national AS
SELECT
    year,
    ROUND(yield_kg_per_ha,    0) AS national_yield_kg_per_ha,
    ROUND(area_harvested_ha,  0) AS national_area_ha,
    ROUND(production_tonnes,  0) AS national_production_tonnes
FROM read_parquet('{SILVER}/silver_fao_national_yield.parquet')
""")

df = q("SELECT * FROM stg_fao_national ORDER BY year DESC LIMIT 5")
print(f"stg_fao_national: {q('SELECT COUNT(*) AS n FROM stg_fao_national')['n'].values[0]} rows")
df

stg_fao_national: 24 rows


,year,national_yield_kg_per_ha,national_area_ha,national_production_tonnes
0,2024,529.0,11448789.0,60570453.0
1,2023,521.0,11637915.0,60601181.0
2,2022,498.0,11600658.0,57776022.0
3,2021,487.0,11666205.0,56772472.0
4,2020,479.0,11693246.0,56056815.0


### 3d. Staging View — District Dimension

Time-invariant district attributes: administrative hierarchy and geographic area.

In [21]:
con.execute(f"""
CREATE OR REPLACE VIEW stg_dim_district AS
SELECT
    district_code,
    division_name,
    district_name,
    ROUND(area_km2, 1) AS area_km2
FROM read_parquet('{SILVER}/silver_dim_district.parquet')
""")

df = q("SELECT * FROM stg_dim_district LIMIT 5")
print(f"stg_dim_district: {q('SELECT COUNT(*) AS n FROM stg_dim_district')['n'].values[0]} districts")
df

stg_dim_district: 64 districts


,district_code,division_name,district_name,area_km2
0,BGD.1.1_1,Barisal,Barguna,1314.0
1,BGD.1.2_1,Barisal,Barisal,2242.6
2,BGD.1.3_1,Barisal,Bhola,1834.7
3,BGD.1.4_1,Barisal,Jhalokati,716.9
4,BGD.1.5_1,Barisal,Patuakhali,2489.4


### 3e. Staging View — BBS Boro Panel (Gap-Filled)

The complete 2001–2022 Boro yield panel with data source provenance flag.

In [22]:
con.execute(f"""
CREATE OR REPLACE VIEW stg_boro_panel AS
SELECT
    district_name,
    year,
    season,
    ROUND(yield_mt_ha, 3)  AS yield_mt_ha,
    ROUND(mean_ndvi,   4)  AS mean_ndvi,
    data_source,
    calibration_r2
FROM read_parquet('{SILVER}/silver_boro_panel_complete.parquet')
WHERE yield_mt_ha IS NOT NULL
""")

df = q("SELECT data_source, COUNT(*) AS n FROM stg_boro_panel GROUP BY data_source")
print("Boro panel by data source:")
print(df.to_string(index=False))

Boro panel by data source:
 data_source   n
interpolated  20
  ndvi_proxy 682
  bbs_actual 731


### 3f. Gold Fact Table — `fact_district_season`

The central analytical table of the research project. One row per district × season × year, joining NDVI, climate, BBS yield, FAO national benchmark, and district geography.

**Table structure:**
- Primary keys: `district_code`, `district_name`, `year`, `season`
- Satellite variables: `ndvi`, `ndvi_anomaly`, `ndvi_long_term_mean`
- Climate variables: `temp_mean_c`, `temp_max_c`, `precip_total_mm`, `precip_anomaly_mm`, `temp_anomaly_c`
- Outcome variable: `boro_yield_mt_ha`, `yield_data_source`
- National benchmark: `national_yield_kg_per_ha`
- Geography: `division_name`, `area_km2`

In [23]:
con.execute("""
CREATE OR REPLACE TABLE fact_district_season AS

WITH ndvi     AS (SELECT * FROM stg_ndvi),
     climate  AS (SELECT * FROM stg_climate),
     nat      AS (SELECT * FROM stg_fao_national),
     dim      AS (SELECT * FROM stg_dim_district),
     boro_yld AS (
         SELECT district_name, year, yield_mt_ha, data_source
         FROM stg_boro_panel
     )

SELECT
    dim.district_code,
    dim.division_name,
    n.district_name,
    n.year,
    n.season,

    dim.area_km2,

    n.ndvi,
    n.ndvi_anomaly,
    n.ndvi_long_term_mean,

    c.temp_mean_c,
    c.temp_max_c,
    c.precip_total_mm,
    c.precip_max_month_mm,
    c.precip_anomaly_mm,
    c.temp_anomaly_c,

    b.yield_mt_ha AS boro_yield_mt_ha,
    b.data_source AS yield_data_source,

    nat.national_yield_kg_per_ha,

    CURRENT_TIMESTAMP AS created_at

FROM ndvi n
LEFT JOIN climate c
       ON n.district_name = c.district_name
      AND n.year = c.year
      AND n.season = c.season

LEFT JOIN dim
       ON n.district_name = dim.district_name

LEFT JOIN nat
       ON n.year = nat.year

LEFT JOIN boro_yld b
       ON n.district_name = b.district_name
      AND n.year = b.year
      AND n.season = 'boro'

WHERE n.year BETWEEN 2001 AND 2023
""")

### 3g. Gold Mart — `mart_vulnerability_index`

The primary research output of the pipeline: a composite agricultural vulnerability score for each of Bangladesh's 64 districts, constructed from four normalised components.

**Index construction:**

| Component | Weight | Operationalisation |
|---|---|---|
| NDVI stress | 35% | 1 − normalised mean Boro NDVI (lower NDVI = higher stress) |
| Precipitation variability | 30% | Coefficient of variation of seasonal precipitation |
| Frequency of poor years | 20% | Fraction of years with NDVI anomaly < −0.05 |
| Heat stress | 15% | Normalised mean seasonal maximum temperature |

Each component is min-max normalised to [0, 1] across districts before weighting. The composite score ranges from 0 (least vulnerable) to 1 (most vulnerable). Districts scoring ≥ 0.70 are classified **High**, 0.40–0.70 **Medium**, and < 0.40 **Low**.

**Weight justification:** The 35% weight on NDVI stress reflects its status as an integrated measure of crop condition that subsumes the effects of water, temperature, and agronomic inputs on vegetation. The 30% weight on precipitation variability captures the dominant climate stressor for rainfed and supplementary-irrigated systems in Bangladesh.

In [24]:
con.execute("""
CREATE OR REPLACE TABLE mart_vulnerability_index AS

WITH fact AS (SELECT * FROM fact_district_season),

-- Step 1: compute raw stress metrics per district
district_stats AS (
    SELECT
        district_name,
        division_name,
        area_km2,

        -- Boro NDVI health (lower = more stressed)
        AVG(CASE WHEN season='boro' THEN ndvi END)                   AS mean_ndvi_boro,
        STDDEV(CASE WHEN season='boro' THEN ndvi END)                AS std_ndvi_boro,

        -- Rainfall variability: coefficient of variation across all seasons
        AVG(precip_total_mm)                                          AS mean_precip,
        STDDEV(precip_total_mm) / NULLIF(AVG(precip_total_mm), 0)    AS cv_precip,

        -- Heat stress proxy
        AVG(temp_max_c)                                               AS mean_temp_max,

        -- Frequency of anomalously poor NDVI years (< -0.05 below district mean)
        SUM(CASE WHEN ndvi_anomaly < -0.05 THEN 1 ELSE 0 END)        AS n_poor_ndvi_years,

        -- Frequency of anomalously high rainfall (flood proxy)
        SUM(CASE WHEN precip_anomaly_mm > 200 THEN 1 ELSE 0 END)     AS n_high_rain_years,

        COUNT(*) AS n_obs
    FROM fact
    GROUP BY district_name, division_name, area_km2
),

-- Step 2: min-max normalise each component to 0–1 scale
normalised AS (
    SELECT
        *,
        -- Precipitation variability: higher CV = more vulnerable
        (cv_precip - MIN(cv_precip) OVER())
            / NULLIF(MAX(cv_precip) OVER() - MIN(cv_precip) OVER(), 0)
                                                                      AS score_precip_var,

        -- NDVI stress: lower Boro NDVI = more vulnerable (inverted)
        1.0 - (mean_ndvi_boro - MIN(mean_ndvi_boro) OVER())
            / NULLIF(MAX(mean_ndvi_boro) OVER() - MIN(mean_ndvi_boro) OVER(), 0)
                                                                      AS score_ndvi_stress,

        -- Frequency of poor years: more = more vulnerable
        (n_poor_ndvi_years - MIN(n_poor_ndvi_years) OVER())
            / NULLIF(MAX(n_poor_ndvi_years) OVER() - MIN(n_poor_ndvi_years) OVER(), 0)
                                                                      AS score_freq_stress,

        -- Heat stress: higher temp max = more vulnerable
        (mean_temp_max - MIN(mean_temp_max) OVER())
            / NULLIF(MAX(mean_temp_max) OVER() - MIN(mean_temp_max) OVER(), 0)
                                                                      AS score_heat_stress
    FROM district_stats
)

-- Step 3: weighted composite score
SELECT
    district_name,
    division_name,
    ROUND(area_km2,          1) AS area_km2,
    ROUND(mean_ndvi_boro,    4) AS mean_ndvi_boro,
    ROUND(mean_precip,       1) AS mean_precip_mm,
    ROUND(cv_precip,         3) AS cv_precip,
    ROUND(mean_temp_max,     2) AS mean_temp_max_c,
    n_poor_ndvi_years,
    n_high_rain_years,

    -- Weights: NDVI stress 35%, precip variability 30%, frequency 20%, heat 15%
    ROUND(
        score_ndvi_stress * 0.35
      + score_precip_var  * 0.30
      + score_freq_stress * 0.20
      + score_heat_stress * 0.15,
    4) AS vulnerability_index,

    CASE
        WHEN (score_ndvi_stress*0.35 + score_precip_var*0.30
            + score_freq_stress*0.20 + score_heat_stress*0.15) >= 0.70 THEN 'High'
        WHEN (score_ndvi_stress*0.35 + score_precip_var*0.30
            + score_freq_stress*0.20 + score_heat_stress*0.15) >= 0.40 THEN 'Medium'
        ELSE 'Low'
    END AS vulnerability_tier

FROM normalised
ORDER BY vulnerability_index DESC
""")

print("mart_vulnerability_index created.")
print()
print("=== Tier distribution ===")
print(q("SELECT vulnerability_tier, COUNT(*) AS n_districts FROM mart_vulnerability_index GROUP BY 1 ORDER BY 1").to_string(index=False))
print()
print("=== Top 10 most vulnerable districts ===")
q("""
SELECT district_name, division_name,
       vulnerability_index, vulnerability_tier,
       n_poor_ndvi_years, ROUND(cv_precip,3) AS cv_precip
FROM mart_vulnerability_index
LIMIT 10
""")

mart_vulnerability_index created.

=== Tier distribution ===
vulnerability_tier  n_districts
              High            4
               Low           10
            Medium           50

=== Top 10 most vulnerable districts ===


,district_name,division_name,vulnerability_index,vulnerability_tier,n_poor_ndvi_years,cv_precip
0,Chapai Nawabganj,Rajshahi,0.7239,High,7.0,0.517
1,Satkhira,Khulna,0.7182,High,0.0,0.641
2,Bhola,Barisal,0.7149,High,3.0,0.560
3,Barguna,Barisal,0.7080,High,4.0,0.609
4,Patuakhali,Barisal,0.6992,Medium,4.0,0.588
5,Pabna,Rajshahi,0.6967,Medium,11.0,0.440
6,Faridpur,Dhaka,0.6788,Medium,9.0,0.493
7,Chuadanga,Khulna,0.6670,Medium,6.0,0.562
8,Kushtia,Khulna,0.6664,Medium,6.0,0.501
9,Meherpur,Khulna,0.6354,Medium,4.0,0.559


### 3h. Gold Layer Verification

Row counts, coverage statistics, and sample data confirm that both analytical tables are correctly populated before the pipeline is declared complete.

In [25]:
print("=== Gold Layer Summary ===")
for table in ["fact_district_season", "mart_vulnerability_index"]:
    n = q(f"SELECT COUNT(*) AS n FROM {table}")["n"].values[0]
    print(f"  {table:<35} {n:>6,} rows")

print("\n=== fact_district_season coverage ===")
print(q("""
SELECT
    season,
    COUNT(DISTINCT district_name) AS districts,
    COUNT(DISTINCT year)          AS years,
    MIN(year)                     AS first_year,
    MAX(year)                     AS last_year,
    ROUND(AVG(ndvi), 3)           AS avg_ndvi,
    COUNT(boro_yield_mt_ha)       AS rows_with_yield
FROM fact_district_season
GROUP BY season ORDER BY season
""").to_string(index=False))

print("\n=== Gold layer ready for Phase 4 GIS analysis ===")

con.close()
print(f"DuckDB connection closed. Database saved at: {DB_PATH}")

=== Gold Layer Summary ===
  fact_district_season                 4,416 rows
  mart_vulnerability_index                64 rows

=== fact_district_season coverage ===
season  districts  years  first_year  last_year  avg_ndvi  rows_with_yield
  aman         64     23        2001       2023     0.508                0
   aus         64     23        2001       2023     0.468                0
  boro         64     23        2001       2023     0.495             1405

=== Gold layer ready for Phase 4 GIS analysis ===
DuckDB connection closed. Database saved at: data\gold\bangladesh_agri.duckdb
